> **How to use this notebook.** Step 6 of 6: sensitivity analysis. Needs: 01 (software, SALib) and the work folder of 02 (`WORK_DIR`: the model's code, the model's configuration and the derived inputs). Every section has a `demo` size (minutes on 2 vCPUs) and states the size of the full design; the full designs are long (the Sobol design is 43,008 model runs on the whole island) and are run outside the notebooks.
>
> **Nurnet files (included).** The site list (`nurnet_nuraghi.shp`, 7,167 nuraghi with tower counts) and the per-site building stone file are Nurnet's data, included in `data/nurnet/`; 02 copies them to the names the configuration expects (`DATA_DIR/sites/`, `DATA_DIR/stone/`) and checks their SHA-256. For the latest version and source information contact Nurnet at nurnet.presidente@gmail.com. Details: [HOW_TO_USE.md](HOW_TO_USE.md#nurnet-files).
>
> Order: 01_setup, 02_inputs, 03_scenarios, 04_analysis_report, 05_audit, 06_sensitivity (Jupyter or Google Colab). Code: the model (`code/`); configuration: `config/scenarios/config_full_audited.yaml` (the model's configuration as the queue reads it; the same values as `config/config_full_audited.yaml`).

# 06 Sensitivity: the A5 grid, Morris screening, Sobol indices

Three sensitivity steps of the project, each run here at a demo size with the published code (lift, every lift
rule, at the undiscounted rate; the learning factor and skill act on quarrying, dressing and building only, never on
lift; ODD 7.9) and the model's configuration:

| section | what | design (full size) | demo here (`SIZE = "demo"`) |
|---|---|---|---|
| 1 | **A5 alliance sensitivity grid** (`model.py --alliance sensitivity`): A5_full with help radius `R_m` {5, 10, 20 km} x progress ratio `b` {0.85, 0.95, 1.0} x track cost {0.5, 2, 5 man-h/m} = 27 cells, plus `A0_alone` as the reference; summary table `sensitivity.csv` (`report_alliance.sensitivity`: island man-hours and % vs A0_alone per cell) | 7,167 sites, seeds 1-100 per cell (`alliance.sensitivity.seeds` in the config: 1-10 by default) | the first `SITES` sites, seed 1 |
| 2 | **Morris screening** of the headline `MOST_REALISTIC_AUDITED_LIT_P10` (`notebooks/sensitivity.py`): 32 factors, elementary effects, mu* and sigma (SALib) | r = 10 trajectories, p = 4 levels, sample seed 20261109: 330 evaluations x seeds 1-3, whole island | r = `MORRIS_R` trajectories, the first `SITES` sites, seed 1 |
| 3 | **Sobol indices** of the headline (Saltelli design, S1 and ST with 95% intervals; SALib) on the 24 factors kept after the Morris screen + `R_m` + `lift_learning` | N = 512, k = 26, sample seed 20261110: 14,336 evaluations x seeds 1-3, whole island | N = `SOBOL_N` base rows, without `lift_learning` (see section 3), the first `SITES` sites, seed 1 |

The demo indices come from a few runs on a few sites: they show that the chain works, not what the full designs
find. Results of the full designs are in `outputs/sensitivity/` (the A5 grid now; Morris, Sobol and feasibility land
with the final pass) and in the results package.
The full designs are regenerated exactly by the same functions (`sensitivity.morris_design(..., r=10)`,
`sensitivity.sobol_design(..., N=512)`); with `FULL_DESIGN_DIR` set to the design files of the results package, the
cells compare them row by row.

## Config (the only cell you edit)

All paths are set at the top of this cell. With `"auto"` the notebook uses the repository layout: `code/`,
`config/`, `data/` (raw inputs, see `data/README.md`), `data/derived/cache/` (derived inputs, see
`data/derived/README.md`) and `outputs/` (everything the notebooks write). Environment variables `NB_REPO_DIR`,
`NB_DATA_DIR`, `NB_DERIVED_DIR`, `NB_CONFIG`, `NB_OUTPUTS_DIR`, `NB_WORK_DIR`, `NB_RESULTS_DIR`, `NB_LOCK`,
`NB_INSTALL` (0 = never install) override the defaults (for headless runs with `jupyter nbconvert --execute`).
`CONFIG` ("auto") is `config/scenarios/config_full_audited.yaml`, the model's configuration: the same
values as the model's configuration `config/config_full_audited.yaml` (audited parameters, minimum haul
`transport.min_haul_flatm: 15`, near-site rule `quarry.near_site_fixed_face_m: 1000`); only `cache_dir` / `output_dir` differ.

**Google Colab:** the repository is cloned to `/content/Repast_Nuraghe` by the setup cell. Each notebook runs in its own Colab runtime, so put the data and outputs on Google Drive: `MOUNT_DRIVE = True`, `DATA_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data"`, `DERIVED_DIR = "/content/drive/MyDrive/Repast_Nuraghe/data/derived/cache"`, `OUTPUTS_DIR = "/content/drive/MyDrive/Repast_Nuraghe/outputs"` (the same values in every notebook).

`SIZE`: `"demo"` (the sizes above) or `"custom"` (your `SITES`, `SEEDS`, `MORRIS_R`, `SOBOL_N`; `SITES = None` = all
7,167 sites, which takes about a minute per evaluation and ~1 GB). `STONES7_CACHE`: the derived-input cache built with
the stones7 rule (the queue's `cache_alt_stones7`, `data/derived/README.md`); without it the factor `stone_rule` is
fixed at hybrid. `FULL_DESIGN_DIR`: a folder with the full design files of the results package (`morris_design.csv`,
`sobol_design.csv`), optional. `RUN_A5`, `RUN_MORRIS`, `RUN_SOBOL` switch the sections.

In [ ]:
import os

# Repository layout: the only lines you may need to change. "auto" = the default folder of the repository.
REPO_DIR = os.environ.get("NB_REPO_DIR", "auto")         # the cloned repository ("auto": the parent of notebooks/)
REPO_URL = "https://github.com/maltaweel/Repast_Nuraghe"  # Colab: cloned to /content/Repast_Nuraghe if not found
DATA_DIR = os.environ.get("NB_DATA_DIR", "auto")         # raw inputs = data_dir ("auto": REPO_DIR/data; see data/README.md)
DERIVED_DIR = os.environ.get("NB_DERIVED_DIR", "auto")   # cached derived inputs ("auto": REPO_DIR/data/derived/cache)
CONFIG = os.environ.get("NB_CONFIG", "auto")             # base config ("auto": REPO_DIR/config/scenarios/config_full_audited.yaml,
                                                         # the model's configuration)
OUTPUTS_DIR = os.environ.get("NB_OUTPUTS_DIR", "auto")   # everything written here ("auto": REPO_DIR/outputs)
WORK_DIR = os.environ.get("NB_WORK_DIR", "auto")         # fast local disk for code + cache copies ("auto": OUTPUTS_DIR/work)
RESULTS_DIR = os.environ.get("NB_RESULTS_DIR", "auto")   # where runs are kept ("auto": OUTPUTS_DIR/notebook_runs)
MOUNT_DRIVE = False                                      # Colab only: mount Google Drive at /content/drive
INSTALL = os.environ.get("NB_INSTALL", "1") == "1"      # install OpenMPI + pinned packages if this runtime lacks them
LOCK = os.environ.get("NB_LOCK") or None                 # optional flock file (shared servers only)

SIZE = os.environ.get("NB_SENS_SIZE", "demo")             # "demo" | "custom"
SITES = 30                                               # custom: first N sites of the haul table (None = all 7,167)
SEEDS = "1"                                              # custom: seeds per evaluation, e.g. "1,2,3" (full designs: 1,2,3)
MORRIS_R = 2                                             # custom: Morris trajectories (full design: 10)
SOBOL_N = 4                                              # custom: Sobol base rows, a power of 2 (full design: 512)
RANKS = int(os.environ.get("NB_RANKS", min(2, os.cpu_count() or 1)))   # MPI ranks for section 1
STONES7_CACHE = os.environ.get("NB_STONES7_CACHE") or None   # stones7 derived-input cache (optional)
FULL_DESIGN_DIR = os.environ.get("NB_FULL_DESIGN_DIR") or None   # full design files of the results package (optional)
RUN_A5, RUN_MORRIS, RUN_SOBOL = True, True, True
LABEL = "sensitivity_" + SIZE                            # results subfolder of RESULTS_DIR

## Setup
Same cell in every notebook: repository (cloned on Colab), folders, install check. Then the work folder of 02
(made here if this runtime has none) and the code version.

In [ ]:
# Setup (no edits needed): find the repository, set the folders, install if needed, import the helpers in repro.py.
import os, sys, glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and MOUNT_DRIVE and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

def _is_repo(d):
    return os.path.isfile(os.path.join(d, "notebooks", "repro.py")) and os.path.isdir(os.path.join(d, "code"))

def _find_repo(p):
    cands = [p] if p != "auto" else [os.path.dirname(os.getcwd()), os.getcwd(), "/content/Repast_Nuraghe"]
    for c in cands:
        if _is_repo(c):
            return os.path.abspath(c)
    if IN_COLAB and p == "auto":                     # Colab opens only the notebook: clone the repository once
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/Repast_Nuraghe"], check=True)
        return "/content/Repast_Nuraghe"
    raise FileNotFoundError("Repository not found (tried " + ", ".join(cands) + "). Set REPO_DIR in the config cell.")

REPO_DIR = _find_repo(REPO_DIR)
PACKAGE_DIR = REPO_DIR                                     # name used by the helper calls below
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))
import repro
if OUTPUTS_DIR == "auto":
    OUTPUTS_DIR = os.path.join(REPO_DIR, "outputs")
OUTPUTS_DIR = os.path.abspath(OUTPUTS_DIR)
L = repro.set_layout(repo=REPO_DIR, data=DATA_DIR, cache=DERIVED_DIR, config=CONFIG,
                     reference=os.path.join(OUTPUTS_DIR, "reference"))
DATA_DIR, DERIVED_DIR, CONFIG = L["data"], L["cache"], L["config"]
WORK_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "work") if WORK_DIR == "auto" else WORK_DIR)
RESULTS_DIR = os.path.abspath(os.path.join(OUTPUTS_DIR, "notebook_runs") if RESULTS_DIR == "auto" else RESULTS_DIR)
for k in ("REPO_DIR", "DATA_DIR", "DERIVED_DIR", "CONFIG", "WORK_DIR", "RESULTS_DIR"):
    print(f"{k:12s}", globals()[k])
status = repro.ensure_env(install=INSTALL)
print("MPI:", repro.mpi_report(), "| Python", sys.version.split()[0], "| CPUs", os.cpu_count())
assert status == "ok", "Restart the kernel now (see message above), then run this notebook again."
import pandas as pd
import sensitivity
pd.set_option("display.width", 200)
if SIZE == "demo":
    SITES, SEEDS, MORRIS_R, SOBOL_N = 30, "1", 2, 4
elif SIZE != "custom":
    raise ValueError(f"SIZE must be demo or custom, not {SIZE!r}")
if not os.path.isfile(os.path.join(WORK_DIR, "config.yaml")):
    print("no WORK_DIR/config.yaml (02_inputs not run in this runtime): making the work folder now")
    repro.prepare_inputs()      # only if inputs are missing (no DERIVED_DIR on Drive): as 02, incl. the ~30-45 min rebuild
    repro.make_workdir(PACKAGE_DIR, WORK_DIR, cache="prebuilt")
print("code version", repro.check_code_version(os.path.join(WORK_DIR, "code")))
OUT = os.path.join(RESULTS_DIR, LABEL)
os.makedirs(OUT, exist_ok=True)
print(f"size {SIZE}: sites {SITES or 'all'}, seeds {SEEDS}, Morris r {MORRIS_R}, Sobol N {SOBOL_N}; results in {OUT}")

## 1. A5 alliance sensitivity grid (`sensitivity.csv`)
The 27 cells `SENS_R<R_m>_b<b>_t<track cost>` of `alliance.sensitivity` in the config (A5_full with the three values
set; `alliance.scenario` resolves them) and `A0_alone` (every community alone, the reference of the % column) are run
with `model.py` and MPI, then the code's own `report_alliance.sensitivity` writes
`RESULTS_DIR/<LABEL>/scenarios_alliance/sensitivity/sensitivity.csv`. In the model, `b` (the Wright progress ratio of
the A3 learning) discounts quarrying, dressing and building, not lift.

**Expected result:** 28 alliance runs (`N to run`), then a table with 27 rows (one per cell): `R_m`, `b`,
`track_cost`, `seeds_run`, island man-hours (mean, min, max), `pct_vs_A0`, `share_received`, `share_track`, alliance
counts and the tower-1 medians.

In [ ]:
if RUN_A5:
    names, G = repro.a5_sensitivity_names(WORK_DIR)
    print(f"{len(names)} cells: R_m {G['R_m']} x b {G['b']} x track cost {G['track_cost_mh_per_m']}; base {G['base']}")
    repro.run_a5_sensitivity(WORK_DIR, RESULTS_DIR, LABEL, SITES or "all", SEEDS, ranks=RANKS, lock=LOCK)
    a5 = repro.a5_sensitivity_table(WORK_DIR, RESULTS_DIR, LABEL, lock=LOCK)
    assert len(a5) == len(names)
    display(a5) if "display" in globals() else print(a5.to_string())
else:
    print("RUN_A5 is False: skipped")

## 2. Morris screening of the headline
The factors are the parameters that the parameter audit (`param_audit/PARAM_AUDIT.md`, rows given) lists as the
authors' assumptions, each with a range (+/-50% where no source gives one) or two levels (a rule vs no rule). The
table below lists them; `sensitivity.apply` sets each one on an in-memory copy of the configuration or of the
headline scenario (`key` column), so no file changes. One evaluation = one design row x `SEEDS`, each seed one run
of the season engine in a single process; outputs per evaluation (mean over the seeds): island years, island
man-hours, transport share. Indices: SALib `morris.analyze` (mu* = mean absolute elementary effect over the full
range of the factor, its 95% bootstrap interval, sigma).

**Expected result:** the design (`r (k + 1)` rows), the evaluation log (`ok` per evaluation), and per output a table
of the factors ranked by mu*.

In [ ]:
if RUN_MORRIS:
    names_m = sensitivity.demo_names("morris", STONES7_CACHE)
    display(sensitivity.factor_table(names_m)) if "display" in globals() else print(sensitivity.factor_table(names_m))
    D_m = sensitivity.morris_design(names_m, r=MORRIS_R)
    print(f"Morris design: k = {len(names_m)}, r = {MORRIS_R}: {len(D_m)} evaluations x seeds {SEEDS}")
    res_m = sensitivity.run_design(WORK_DIR, D_m, os.path.join(OUT, "morris_design.csv"), os.path.join(OUT, "morris_results.csv"),
                                   seeds=SEEDS, first_n=SITES, stones7_cache=STONES7_CACHE, lock=LOCK)
    Y_m = sensitivity.per_eval(res_m, D_m)
    I_m = sensitivity.morris_indices(D_m, Y_m, names_m)
    I_m.to_csv(os.path.join(OUT, "morris_indices.csv"), index=False)
    for o in sensitivity.OUTPUTS:
        print(o); display(I_m[I_m.output == o].head(10)) if "display" in globals() else print(I_m[I_m.output == o].head(10).to_string())
else:
    print("RUN_MORRIS is False: skipped")

**Full Morris design.** `sensitivity.morris_design(sensitivity.MORRIS_NAMES, r=10)` (32 factors, sample seed
20261109) is the design of the full screen: 330 evaluations, run with seeds 1, 2, 3 on the whole island. With
`FULL_DESIGN_DIR` set, the next cell compares it with the design file of the results package.

In [ ]:
D_full_m = sensitivity.morris_design(sensitivity.MORRIS_NAMES, r=sensitivity.MORRIS_FULL["r"])
print(f"full Morris design: {len(D_full_m)} evaluations x seeds {sensitivity.MORRIS_FULL['seeds']}")
if FULL_DESIGN_DIR:
    assert sensitivity.compare_design(D_full_m, os.path.join(FULL_DESIGN_DIR, "morris_design.csv"), sensitivity.MORRIS_NAMES)

## 3. Sobol indices of the headline
The 24 factors kept after the Morris screen, plus `R_m` (help radius of the scenario, 5-15 km) and `lift_learning`
(level 0 = lift undiscounted, the model's rule; level 1 = lift discounted with the build stage, the earlier rule).
Fixed at the headline: the 8 other Morris factors and `alliance.alliance_threshold`. Design: SALib `sobol.sample`
(Saltelli, scrambled Sobol sequence, no second-order terms): `N (k + 2)` evaluations; indices: SALib `sobol.analyze`
(S1 first order, ST total, 95% bootstrap intervals). Integer factors use equal-probability bins.

`lift_learning` level 1 needs a code switch (`sobol.lift_learning`) that is **not in the published code**: the full
Sobol run used a copy of `code/` with that one switch added (default off = `code/` exactly; `sensitivity/sobol2/`). The demo
therefore leaves it out (k = 25, or 24 without the stones7 cache). With so few base rows the indices are very
uncertain (wide intervals, values outside [0, 1] are possible); that is expected at demo size.

**Expected result:** the design (`N (k + 2)` rows), the evaluation log, and per output a table of S1 and ST.

In [ ]:
if RUN_SOBOL:
    names_s = sensitivity.demo_names("sobol", STONES7_CACHE)
    D_s = sensitivity.sobol_design(names_s, N=SOBOL_N)
    print(f"Sobol design: k = {len(names_s)}, N = {SOBOL_N}: {len(D_s)} evaluations x seeds {SEEDS}")
    res_s = sensitivity.run_design(WORK_DIR, D_s, os.path.join(OUT, "sobol_design.csv"), os.path.join(OUT, "sobol_results.csv"),
                                   seeds=SEEDS, first_n=SITES, stones7_cache=STONES7_CACHE, lock=LOCK)
    Y_s = sensitivity.per_eval(res_s, D_s)
    I_s = sensitivity.sobol_indices(Y_s, names_s)
    I_s.to_csv(os.path.join(OUT, "sobol_indices.csv"), index=False)
    for o in sensitivity.OUTPUTS:
        print(o); display(I_s[I_s.output == o].head(10)) if "display" in globals() else print(I_s[I_s.output == o].head(10).to_string())
else:
    print("RUN_SOBOL is False: skipped")

**Full Sobol design.** `sensitivity.sobol_design(sensitivity.SOBOL_NAMES, N=512)` (26 factors, sample seed
20261110) is the design of the full run: 14,336 evaluations x seeds 1, 2, 3 = 43,008 model runs on the whole island
(about 48 s per evaluation and 1.1 GB on one core). With `FULL_DESIGN_DIR` set, the next cell compares it with the
design file of the results package. To run it you need the code switch for `lift_learning` (see above) or a design
without that factor.

In [ ]:
D_full_s = sensitivity.sobol_design(sensitivity.SOBOL_NAMES, N=sensitivity.SOBOL_FULL["N"])
print(f"full Sobol design: {len(D_full_s)} evaluations x seeds {sensitivity.SOBOL_FULL['seeds']}")
if FULL_DESIGN_DIR:
    assert sensitivity.compare_design(D_full_s, os.path.join(FULL_DESIGN_DIR, "sobol_design.csv"), sensitivity.SOBOL_NAMES)
print("Sensitivity notebook done. Outputs:", OUT)